## Review and Approve Genie Spaces

This notebook allows you to review the discovered Genie Spaces and select which ones to migrate.

**What it does:**
1. Loads the inventory from Src_01
2. Displays all discovered spaces
3. Allows you to filter/select spaces
4. Saves the approved list for export

**Manual step:** Run this notebook interactively and type CONFIRM when ready.

## Setup

In [ ]:
import os
import pandas as pd

## Read Parameters

In [ ]:
dbutils.widgets.text("volume_path", "", "Export Volume Path")

volume_path = dbutils.widgets.get("volume_path")

if not volume_path:
    raise ValueError("volume_path parameter is required")

print(f"Volume path: {volume_path}")

## Load Inventory

In [ ]:
inventory_path = os.path.join(volume_path, "genie_inventory", "genie_inventory.csv")

if not os.path.exists(inventory_path):
    raise FileNotFoundError(
        f"Inventory file not found: {inventory_path}\n"
        "Run Src_01_Inventory_Generation first."
    )

df = pd.read_csv(inventory_path)
print(f"Loaded {len(df)} spaces from inventory")

## Display All Spaces

In [ ]:
print("=" * 60)
print("DISCOVERED GENIE SPACES")
print("=" * 60)
display(df)

## Filter Spaces (Optional)

Uncomment and modify the filter below to exclude specific spaces.

In [ ]:
# Example filters (uncomment to use):

# Filter by title pattern
# df = df[df['title'].str.contains('Sales', case=False, na=False)]

# Filter by parent path
# df = df[df['parent_path'].str.startswith('/Workspace/Shared')]

# Exclude specific spaces by title
# exclude_titles = ['Test Space', 'Dev Space']
# df = df[~df['title'].isin(exclude_titles)]

# Only include spaces with benchmarks
# df = df[df['benchmark_count'] > 0]

print(f"Spaces after filtering: {len(df)}")

## Review Filtered Spaces

In [ ]:
print("=" * 60)
print("SPACES TO BE MIGRATED")
print("=" * 60)
display(df[["space_id", "title", "benchmark_count", "parent_path"]])
print(f"\nTotal: {len(df)} space(s)")
print(f"Total benchmarks: {df['benchmark_count'].sum()}")

## Confirm Selection

**IMPORTANT:** Review the spaces above carefully. Type `CONFIRM` below to approve them for export.

In [ ]:
dbutils.widgets.text("confirmation", "", "Type CONFIRM to approve")

confirmation = dbutils.widgets.get("confirmation")

if confirmation.strip().upper() != "CONFIRM":
    print("\n⚠️  NOT CONFIRMED")
    print("Set the 'confirmation' widget to CONFIRM to proceed.")
    print("Review the spaces above and type CONFIRM when ready.")
    dbutils.notebook.exit("Not confirmed - review and set confirmation=CONFIRM")
else:
    print("✓ Confirmed! Proceeding to save approved inventory...")

## Save Approved Inventory

In [ ]:
approved_dir = os.path.join(volume_path, "genie_inventory_approved")
os.makedirs(approved_dir, exist_ok=True)

approved_path = os.path.join(approved_dir, "inventory_approved.csv")

df.to_csv(approved_path, index=False)

print(f"\n✓ Approved inventory saved to: {approved_path}")
print(f"  Spaces approved: {len(df)}")

## Summary

In [ ]:
print("=" * 60)
print("APPROVAL COMPLETE")
print("=" * 60)
print(f"Spaces approved:   {len(df)}")
print(f"Total benchmarks:  {df['benchmark_count'].sum()}")
print(f"Approved file:     {approved_path}")
print("\nNext step: Run src_genie_export job to export the approved spaces.")